In [ ]:
from pathlib import Path
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio

**Nomenclature**

An "`experiment`" consists of multiple "`buffer`s", which in turn consist of multiple "`signal`s". Each signal is an array of PMT voltage values over time saved by the PicoScope if the signal meets the "trigger" criteria.

This notebook assumes that each raw `buffer` (stored as a PSData file, .psdata) has already been converted into a directory containing its constituent `signal`s as MATLAB (.mat) files. The utility functions herein assist in converting these `signal`s into parquet files for easier storage and future processing.

The buffer and signal files must also follow the standard naming conventions:

- Each buffer directory should be named as `<date>-<buffer_number>`.

    - e.g., `20220730-0001` is the first buffer from July 30th, 2022.

- Each signal file should be named as `<date>-<buffer_num>_<signal_num>.mat`.

    - e.g., `20220730-0001_1234.mat` is the 1,234th signal from the first buffer from July 30th, 2022.
    
**Experiment directory structure**

To simplify data processing and analysis, a standard format has been chosen for a "neutron" experiment.

```
<YYYYMMDD>_<Descriptor>\
  |-- processed_data\
    |-- parquets
  |-- raw_data\
    |-- matlab\
      |-- <YYYYMMDD>-0001/
        |-- <YYYYMMDD>-0001_00001.mat
      |-- <YYYYMMDD>-0002/
      |-- ...
    |-- psdata
      |-- *.psdata
```

In [ ]:
# Utility functions

def parse_signal_id(filename: str) -> str:
    """Returns a string in the format 'b<buffer number>s<signal number>',
    which is parsed from `filename`, assuming the file is named in the format:
    
    <date (YYYYMMDD)>-<buffer number>_<signal number>.<ext>
    
    Example: '20220730-0001_1234.mat' -> 'b0001s1234'
    """
    bufnum, signum = re.split('_|-|\.', filename)[1:3]
    return f"b{bufnum}s{signum}"

def get_signal_data(filepath: Path, key: str='A') -> np.array:
    """Returns a `numpy.array` of the data ('A' key) from a .mat file."""
    # Load a .mat file into a dictionary
    mat = sio.loadmat(filepath)
    
    if key not in mat.keys():
        print(f"Key \x1b[31m'{key}'\x1b[0m not found! Try one of:")
        print(*list(mat.keys()), sep='\n')
    else:
        return mat[key].flatten()
    
def convert_mats_to_df(mat_dir: Path, sort_signals: bool = False) -> pd.DataFrame:
    """Loads all .mat files from `mat_dir` and converts them to rows in a
    `pandas.DataFrame`."""
    mat_files = [f for f in mat_dir.iterdir() if f.suffix == '.mat']
    if sort_signals:
        mat_files.sort()
    
    # Create empty dictionary to store signals
    temp_dict = {}
    for file in mat_files:
        signal_id = parse_signal_id(file.name)
        signal_data = get_signal_data(file)
        temp_dict[signal_id] = signal_data
    
    df = pd.DataFrame.from_dict(temp_dict)
    
    return df.T
    
def convert_mats_to_parquet(
    mat_dir: Path, out_dir: Path, sort_signals: bool = False, verbose: bool = False
) -> None:
    """Convert all `.mat` files in `mat_dir` into a single `.parquet` file."""

    # Create the full path for the output parquet file
    out_path = out_dir / f"{mat_dir.name}.parquet"

    # TODO: handle overwrite more gracefully
    if out_path.exists():
        if verbose:
            print(
                f"The file \x1b[33m'{out_path.name}'\x1b[0m already exists."
                " Overwriting..."
            )

    if verbose:
        start = time.time() # Record conversion start time

    # Create a DataFrame with all signals
    df = convert_mats_to_df(mat_dir, sort_signals = sort_signals)
    
    # Parquets require strings for column names
    df.columns = df.columns.astype(str)
    df.to_parquet(out_path)

    if verbose:
        print(
            f"'{mat_dir}' -> '{out_path}'\n"
            f"Finished in \x1b[32m{time.time() - start:.3f}\x1b[0m s."
        )

## Examples

### Convert all signals from a buffer directory to a single parquet file

Converts every signal (MATLAB) file in a directory to a row in a parquet file. Each signal is named according to the buffer number and the signal number contained in the filename.

The cells below will convert the 5,000 `signal`s contained in the `20220730-0001` buffer into rows of a single parquet file, `20220730-0001.parquet`.

In [ ]:
# Path to directory that contains signals stored as .mat files
BUFFER_DIR = Path('../sample_dataset/raw_data/mat/20220730-0001/')
PARQ_OUT_DIR = Path('../sample_dataset/processed_data/parquets/')

In [ ]:
convert_mats_to_parquet(
    BUFFER_DIR,
    PARQ_OUT_DIR,
    sort_signals=True,
    verbose=True,
)

### Convert signals from a list of buffer directories to a series of parquet files

In [ ]:
BUFFER_ROOT_DIR = Path('../sample_dataset/raw_data/mat/')
PARQ_OUT_DIR = Path('../sample_dataset/processed_data/parquets/')

mat_dirs = BUFFER_ROOT_DIR.iterdir()

In [ ]:
start = time.time()
buff_count = 0
for buffer_dir in mat_dirs:
    buff_count += 1
    convert_mats_to_parquet(
        buffer_dir,
        PARQ_OUT_DIR,
        sort_signals=True,
        verbose=False,
    )
    
print(f"Converted \x1b[1;36m{buff_count}\x1b[0m buffers to parquet in \x1b[32m{time.time() - start:.2f}\x1b[1;0m seconds.")

## Convert signals from a list of buffer directories to a single parquet file

In [ ]:
# TODO: add code to concatenate multiple buffers into a single parquet file

In [ ]:
# Plot signals from a parquet file to verify data integrity

fig, ax = plt.subplots(figsize=(15,8))

temp_df = pd.read_parquet("../sample_dataset/processed_data/parquets/20220730-0001.parquet")
temp_df.columns = temp_df.columns.astype(int)
temp_df = temp_df.T

ax.plot(
    temp_df
)

fig.show()